# Predictive baselines — named cost models

QPPNet, Zero-Shot Cost Model, Reqo and NNGP.

- **QPPNet** and **Zero-Shot** are point estimators (Tier-1): they fill only
  the point-error sheet; CRPS / interval / uncertainty sheets are **n/a**.
- **Reqo** and **NNGP** produce a Gaussian predictive distribution and fill all
  four sheets.

These consume structured plan graphs via `build_graph_dataset` (QPPNet /
Zero-Shot / Reqo) or the shared feature vectors (NNGP). See each model's
docstring for the lakehouse feature-gap caveats (esp. Zero-Shot, which cannot
reproduce true zero-shot transfer without catalog statistics).

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.config import *

from uncertainty_prediction.baselines.predictive.common import (
    build_feature_dataset,
    build_graph_dataset,
    evaluate_gaussian_predictions,
    evaluate_point_predictions,
    print_metric_headers_for_excel,
    print_metrics_for_excel,
)
from uncertainty_prediction.baselines.predictive.qppnet import QPPNetBaseline
from uncertainty_prediction.baselines.predictive.zero_shot import ZeroShotBaseline
from uncertainty_prediction.baselines.predictive.reqo import ReqoBaseline
from uncertainty_prediction.baselines.predictive.nngp import NNGPBaseline

import torch

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

train_qids, test_qids = split_query_ids(common, seed=SEED, test_frac=TEST_FRAC)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

# structured plan graphs (QPPNet / Zero-Shot / Reqo)
gdata = build_graph_dataset(
    plans_by_query=plans_by_query, runs_by_query=runs_by_query,
    train_qids=train_qids, test_qids=test_qids, xcol=XCOL, runtime_mode="mean",
)
# flat feature vectors (NNGP)
fdata = build_feature_dataset(
    plans_by_query=plans_by_query, runs_by_query=runs_by_query,
    train_qids=train_qids, test_qids=test_qids, xcol=XCOL, runtime_mode="mean",
)
print("num_ops:", gdata["num_ops"], "| cont_dim:", gdata["cont_dim"], "| feat_dim:", fdata["feature_dim"])

y_test_log = gdata["y_test_log"]
y_test_runtime = np.exp(y_test_log)

## QPPNet (point)

In [ ]:
qpp = QPPNetBaseline(num_ops=gdata["num_ops"], cont_dim=gdata["cont_dim"], device=device, seed=42)
qpp.fit(gdata["train_graphs"], gdata["y_train_log"], num_epochs=100, lr=1e-3, verbose=True)

mu_log = qpp.predict_point_log(gdata["test_graphs"])
qpp_metrics = evaluate_point_predictions(np.exp(mu_log), y_test_runtime)
print_metric_headers_for_excel(qpp_metrics)
print_metrics_for_excel(qpp_metrics)

## Zero-Shot Cost Model (point)

In [ ]:
zs = ZeroShotBaseline(num_ops=gdata["num_ops"], cont_dim=gdata["cont_dim"], device=device, seed=42)
zs.fit(gdata["train_graphs"], gdata["y_train_log"], num_epochs=100, lr=1e-3, verbose=True)

mu_log = zs.predict_point_log(gdata["test_graphs"])
zs_metrics = evaluate_point_predictions(np.exp(mu_log), y_test_runtime)
print_metric_headers_for_excel(zs_metrics)
print_metrics_for_excel(zs_metrics)

## Reqo (Gaussian)

In [ ]:
reqo = ReqoBaseline(num_ops=gdata["num_ops"], cont_dim=gdata["cont_dim"], device=device, seed=42)
reqo.fit(gdata["train_graphs"], gdata["y_train_log"], num_epochs=100, lr=1e-3, verbose=True)

pred = reqo.predict_gaussian(gdata["test_graphs"])
reqo_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], y_test_log)
print_metric_headers_for_excel(reqo_metrics)
print_metrics_for_excel(reqo_metrics)

## NNGP (Gaussian)
Uses the flat feature vectors (`fdata`). NNGP test targets align with `fdata` ordering.

In [ ]:
nngp = NNGPBaseline(depth=3, seed=42)
nngp.fit(fdata["X_train"], fdata["y_train_log"])

pred = nngp.predict_gaussian(fdata["X_test"])
nngp_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], fdata["y_test_log"])
print_metric_headers_for_excel(nngp_metrics)
print_metrics_for_excel(nngp_metrics)

## Side-by-side summary
(point-only models show NaN for the distribution columns)

In [ ]:
summary = pd.DataFrame(
    {
        "QPPNet": qpp_metrics,
        "Zero-Shot": zs_metrics,
        "Reqo": reqo_metrics,
        "NNGP": nngp_metrics,
    }
).T
cols = ["mae", "rmse", "median_q_error", "crps", "cov@50", "cov@90", "cov@99", "mpiw", "unc_spearman", "unc_pearson"]
summary.reindex(columns=cols)